[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/36_explainable_ai/90_exercises.ipynb)

# Bài tập — Chương 36: Explainable AI (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/36_explainable_ai

In [ ]:
import itertools
from math import comb, factorial

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_customer_churn

raw = make_customer_churn(20000, seed=0)
X = pd.DataFrame({
    "tenure": raw.tenure_months, "monthly": raw.monthly_charges, "total": raw.total_charges,
    "support_calls": raw.support_calls, "month_to_month": (raw.contract == "month-to-month").astype(int),
    "two_year": (raw.contract == "two_year").astype(int), "fiber": (raw.internet_service == "fiber").astype(int),
    "ewallet": (raw.payment_method == "e-wallet").astype(int), "senior": raw.senior.astype(int),
    "n_products": raw.n_products,
}).astype(float)
y = raw.churn.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(X_train, y_train)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Giải thích toàn cục và cục bộ; model-agnostic. (b) Vì sao MDI thiên lệch? (c) PDP vs ALE. (d) Định nghĩa giá trị Shapley và tính chất hiệu quả. (e) LIME hoạt động thế nào; điểm yếu? (f) Vì sao giải thích mô hình không phải giải thích nhân quả?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một ngân hàng gửi cho khách hàng bị từ chối vay một thư giải thích sinh tự động bằng SHAP: "Thu nhập của bạn đóng góp −0.31, khu vực sinh sống −0.45". Họ (1) dùng dữ liệu nền là **khách hàng đã được duyệt**; (2) khẳng định mô hình công bằng vì "đã loại bỏ giới tính và dân tộc"; (3) giải thích từ **một** mô hình được chọn trong 20 mô hình có AUC gần như bằng nhau; (4) không đề xuất khách hàng cần làm gì. Đánh giá và đề xuất.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Trò chơi 2 người chơi: $v(\emptyset) = 0$, $v(\{1\}) = 4$, $v(\{2\}) = 2$, $v(\{1, 2\}) = 10$. Tính `phi_1`, `phi_2`.

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
phi_1 = None
phi_2 = None
### END CODE HERE ###
print(phi_1, phi_2)

In [ ]:
check_close([phi_1, phi_2], [6.0, 4.0], message="trung bình đóng góp biên qua 2 thứ tự")
check_close(phi_1 + phi_2, 10 - 0, message="hiệu quả: tổng = v(F) − v(∅)")

**Câu 4.** Mô hình tuyến tính $f(x) = 1 + 2x_1 - x_2$; trung bình nền $\mathbb E[x] = (1, 2)$; điểm $x = (3, 1)$. Tính vector Shapley `phi4` và kiểm tra tổng.

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
phi4 = None
### END CODE HERE ###
print(phi4)

In [ ]:
check_close(phi4, [4.0, 1.0], message="φ_j = β_j (x_j − E x_j)")
check_close(phi4.sum(), (1 + 2 * 3 - 1) - (1 + 2 * 1 - 2), message="tổng = f(x) − f(E[x]) (mô hình tuyến tính)")

## L3 — Code

**Bài 5.** Cài `my_permutation_importance(predict_score, X, y, n_repeats, seed)`: điểm gốc trừ điểm sau khi xáo trộn từng cột (trung bình `n_repeats` lần). Áp dụng với ROC-AUC cho mô hình boosting trên tập test; so với `sklearn.inspection.permutation_importance`.

In [ ]:
def my_permutation_importance(score_fn, X, y, n_repeats=5, seed=0):
    ### START CODE HERE ### (≈ 9 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
auc_score = lambda Xs, ys: roc_auc_score(ys, model.predict_proba(Xs)[:, 1])
mine5 = my_permutation_importance(auc_score, X_test, y_test)
ref5 = pd.Series(permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0).importances_mean, index=X.columns)
print(pd.DataFrame({"tự cài": mine5, "sklearn": ref5}).round(4).sort_values("sklearn", ascending=False))
check_less((mine5 - ref5).abs().max(), 0.01, "khớp sklearn (sai khác do xáo trộn ngẫu nhiên)")
check_true(set(mine5.nlargest(2).index) == set(ref5.nlargest(2).index), "cùng top-2")
check_greater(mine5.rank().corr(ref5.rank()), 0.8, "thứ hạng tương quan cao (Spearman)")

**Bài 6.** Cài `my_pdp(predict, X, feature, grid)` (brute force: đặt cột = mỗi giá trị lưới cho mọi dòng, lấy trung bình). Kiểm tra khớp `sklearn.inspection.partial_dependence(method="brute")` trên lưới mà sklearn trả về.

In [ ]:
def my_pdp(predict, X, feature, grid):
    ### START CODE HERE ### (≈ 1 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
sample6 = X_test.sample(500, random_state=0)
ref6 = partial_dependence(model, sample6, ["tenure"], grid_resolution=15, method="brute", kind="average", response_method="predict_proba")
grid6 = ref6["grid_values"][0]
mine6 = my_pdp(lambda D: model.predict_proba(D)[:, 1], sample6, "tenure", grid6)
check_close(mine6, ref6["average"][0], tol=1e-10, message="khớp sklearn")
check_less(mine6[-1], mine6[0], "rời bỏ giảm theo tenure")

## L4 — Debug

**Bài 7.** Một nhóm dùng rừng ngẫu nhiên **rất sâu** và tính permutation importance để chọn đặc trưng. Họ thêm cột `customer_number` (mã khách hàng ngẫu nhiên) "để thử" và thấy nó **quan trọng**. Tìm lỗi trong cách tính và sửa `importance_for_selection`.

In [ ]:
X7_train = X_train.assign(customer_number=np.random.default_rng(0).permutation(len(X_train)))
X7_test = X_test.assign(customer_number=np.random.default_rng(1).permutation(len(X_test)))
deep_forest = RandomForestClassifier(200, min_samples_leaf=1, random_state=0, n_jobs=-1).fit(X7_train, y_train)


def importance_buggy():
    r = permutation_importance(deep_forest, X7_train, y_train, scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1)
    return pd.Series(r.importances_mean, index=X7_train.columns)


def importance_for_selection():
    ### START CODE HERE ### (≈ 2 dòng)
    return importance_buggy()
    ### END CODE HERE ###


imp_bug, imp_fix = importance_buggy(), importance_for_selection()
print(pd.DataFrame({"trên train (lỗi)": imp_bug, "trên test (đúng)": imp_fix}).round(4))

In [ ]:
check_greater(imp_bug["customer_number"], 0.01, "tái hiện: trên train, mã khách hàng ngẫu nhiên có vẻ quan trọng")
check_less(abs(imp_fix["customer_number"]), 0.005, "trên dữ liệu chưa thấy: ≈ 0")

## L5 — Phân tích

**Bài 8.** **Tương quan chia độ quan trọng.** Dữ liệu tổng hợp: $y = x_1 + 0.5x_3 + \varepsilon$; $x_2 = \rho x_1 + \sqrt{1-\rho^2}\,u$ (tương quan $\rho$ với $x_1$ nhưng **không** ảnh hưởng $y$). Với $\rho \in \{0, 0.5, 0.9, 0.99\}$: train `RandomForestRegressor(200, max_features=1, min_samples_leaf=5)`, lưu permutation importance (tập test, $R^2$) của $x_1, x_2, x_3$ vào `imp8[rho]`; đồng thời hệ số hồi quy tuyến tính `coef8[rho]`. Nhận xét.

In [ ]:
imp8, coef8 = {}, {}
### START CODE HERE ### (≈ 14 dòng)
# với mỗi ρ: sinh 4 000 mẫu (seed 0), chia train/test 70/30, train rừng + LinearRegression, tính permutation importance
pass
### END CODE HERE ###
print("permutation importance (rừng):\n", pd.DataFrame(imp8).round(3))
print("hệ số hồi quy tuyến tính:\n", pd.DataFrame(coef8).round(3))

In [ ]:
check_less(imp8[0.0]["x2"], 0.02, "ρ = 0: x2 không quan trọng")
check_greater(imp8[0.99]["x2"], imp8[0.0]["x2"] + 0.05, "ρ cao: rừng 'chia' độ quan trọng cho x2 dù x2 không tác động")
check_less(imp8[0.99]["x1"], imp8[0.0]["x1"], "ρ cao: độ quan trọng của x1 bị chia bớt")
check_less(abs(coef8[0.99]["x2"]), 0.15, "hồi quy tuyến tính vẫn gần đúng x2 ≈ 0 (đủ dữ liệu)")

## L6 — Thực tế

**Bài 9.** **Kiểm toán công bằng.** Dùng dữ liệu tín dụng của bài 03 (cho sẵn — có thêm nhãn "trả nợ được thật" `repaid`). Mô hình logistic dùng `income`, `district` dự đoán `approved` lịch sử. (1) Viết `group_report(decision, group, repaid)` trả về dict: tỉ lệ duyệt mỗi nhóm, **chênh lệch tỉ lệ duyệt** (demographic parity difference), và **TPR** mỗi nhóm (trong số người thật sự trả được nợ, tỉ lệ được duyệt — equal opportunity). (2) So sánh 3 phương án: (a) mô hình gốc, ngưỡng 0.5; (b) bỏ `district`; (c) mô hình gốc nhưng **ngưỡng riêng cho nhóm A** sao cho TPR hai nhóm gần bằng nhau. Lưu `reports9[tên]` và accuracy (dự đoán `repaid`) của từng phương án vào `acc9[tên]`.

In [ ]:
rng9 = np.random.default_rng(5)
n9 = 20000
group_a = rng9.random(n9) < 0.3
district = np.where(group_a, rng9.random(n9) < 0.8, rng9.random(n9) < 0.15).astype(int)
income = rng9.normal(50 - 8 * group_a, 10, n9)
repaid = (rng9.random(n9) < 1 / (1 + np.exp(-0.1 * (income - 45)))).astype(int)        # khả năng trả nợ thật: chỉ phụ thuộc thu nhập
approved = np.where(group_a & (rng9.random(n9) < 0.3), 0, repaid)                        # lịch sử: 30% hồ sơ tốt của nhóm A vẫn bị từ chối
credit = pd.DataFrame({"income": income, "district": district})
idx_tr, idx_te = train_test_split(np.arange(n9), test_size=0.3, random_state=0)
full = LogisticRegression().fit(credit.iloc[idx_tr], approved[idx_tr])
no_district = LogisticRegression().fit(credit.iloc[idx_tr][["income"]], approved[idx_tr])
p_full = full.predict_proba(credit.iloc[idx_te])[:, 1]
p_nod = no_district.predict_proba(credit.iloc[idx_te][["income"]])[:, 1]
g_te, rep_te = group_a[idx_te], repaid[idx_te]


def group_report(decision, group, repaid):
    ### START CODE HERE ### (≈ 7 dòng)
    return {}
    ### END CODE HERE ###


reports9, acc9 = {}, {}
### START CODE HERE ### (≈ 10 dòng)
# (a) p_full > 0.5; (b) p_nod > 0.5; (c) tìm ngưỡng nhóm A (lưới 0.05–0.5) để |TPR_A − TPR_khác| nhỏ nhất
# reports9[tên] = group_report(...); acc9[tên] = accuracy so với rep_te
best_t = None   # ngưỡng riêng của nhóm A ở (c)
### END CODE HERE ###
print(pd.DataFrame(reports9).round(3))
print("accuracy dự đoán khả năng trả nợ thật:", {k: round(v, 3) for k, v in acc9.items()}, f"| ngưỡng nhóm A = {best_t:.2f}")

In [ ]:
manual = (p_full > 0.5)[g_te].mean()
check_close(reports9["(a) gốc"]["duyệt nhóm A"], manual, message="tỉ lệ duyệt nhóm A")
check_greater(reports9["(a) gốc"]["chênh lệch duyệt"], 0.25, "mô hình gốc: chênh lệch lớn")
check_less(abs(reports9["(c) ngưỡng riêng nhóm A"]["TPR nhóm A"] - reports9["(c) ngưỡng riêng nhóm A"]["TPR nhóm khác"]), 0.05, "(c): TPR gần bằng nhau")
check_less(reports9["(c) ngưỡng riêng nhóm A"]["chênh lệch duyệt"], reports9["(a) gốc"]["chênh lệch duyệt"] - 0.2, "(c): chênh lệch tỉ lệ duyệt giảm mạnh")
check_greater(acc9["(c) ngưỡng riêng nhóm A"], acc9["(a) gốc"] - 0.02, "(c): accuracy so với khả năng trả nợ THẬT gần như không đổi")

## L7 — Nâng cao

**Bài 10.** **KernelSHAP from scratch.** Giá trị Shapley là nghiệm của một bài toán **hồi quy tuyến tính có trọng số** trên các tập con $z \in \{0,1\}^d$ (Lundberg & Lee, 2017) với **Shapley kernel** $\pi(z) = \frac{d - 1}{\binom{d}{|z|}\,|z|\,(d - |z|)}$ và ràng buộc $\phi_0 = v(\emptyset)$, $\sum\phi_j = v(F) - v(\emptyset)$. Cài `kernel_shap(value_fn, d, coalitions)`: (cách đơn giản) đặt trọng số rất lớn (1e6) cho tập rỗng và tập đầy đủ, fit hồi quy tuyến tính có trọng số $v(z) \approx \phi_0 + \sum_j\phi_jz_j$. So với Shapley chính xác khi dùng **mọi** tập con, và khi chỉ lấy mẫu 200 tập con.

In [ ]:
features10 = list(X.columns)
d10 = len(features10)
x10 = X_test.iloc[[5]]
bg10 = X_train.sample(100, random_state=0)
f10 = model.decision_function
value_cache = {}


def value_fn(z):
    key = tuple(int(v) for v in z)
    if key not in value_cache:
        Z = bg10.copy()
        for j in np.flatnonzero(z):
            Z[features10[j]] = x10[features10[j]].values[0]
        value_cache[key] = f10(Z).mean()
    return value_cache[key]


all_coalitions = np.array(list(itertools.product([0, 1], repeat=d10)))
phi_exact = np.zeros(d10)                                              # Shapley chính xác (định nghĩa)
for z in all_coalitions:
    for j in np.flatnonzero(z == 0):
        z_with = z.copy()
        z_with[j] = 1
        s = z.sum()
        phi_exact[j] += factorial(s) * factorial(d10 - s - 1) / factorial(d10) * (value_fn(z_with) - value_fn(z))


def kernel_shap(value_fn, d, coalitions):
    ### START CODE HERE ### (≈ 9 dòng)
    return None
    ### END CODE HERE ###


phi_kernel_all = kernel_shap(value_fn, d10, all_coalitions)
rng10 = np.random.default_rng(0)
sampled = np.vstack([np.zeros(d10, int), np.ones(d10, int), all_coalitions[rng10.choice(len(all_coalitions), 200, replace=False)]])
phi_kernel_200 = kernel_shap(value_fn, d10, sampled)
print(pd.DataFrame({"chính xác": phi_exact, "KernelSHAP (mọi tập con)": phi_kernel_all, "KernelSHAP (200 tập con)": phi_kernel_200},
                   index=features10).round(4))

In [ ]:
check_close(phi_kernel_all, phi_exact, tol=1e-3, message="KernelSHAP với mọi tập con = Shapley chính xác")
check_close(phi_kernel_all.sum(), value_fn(np.ones(d10)) - value_fn(np.zeros(d10)), tol=1e-3, message="hiệu quả")
check_less(np.abs(phi_kernel_200 - phi_exact).max(), 0.15, "lấy mẫu 200/1 024 tập con: xấp xỉ khá tốt")